In [19]:
from utils.common import prepare_dataset, data_split, train_catboost, calculate_metrics, get_most_important_features, make_lags_dataset

df_small = prepare_dataset('../trading_small_df.csv',
                    exclude_cols=['cred_limit', 'fin_cond_index', 'tax_regime', 'reg_date', 'Unnamed: 0'], 
                    year_col='year',
                    id_col='vat_num',
                    dflt_col='dflt_year',
                    drop_fin_zeroes=True,
                    drop_ones_after_ones=True,
                    year_from=2022)
df_small_lag2 = make_lags_dataset(df_small, [1, 2])
vat_nums = df_small_lag2.index.get_level_values('vat_num')

df_small_lag1 = make_lags_dataset(df_small, [1])
df_small_lag1_2024 = df_small_lag1[df_small_lag1.index.get_level_values('year') == 2024]
df_small_lag1_2024_right_nums = df_small_lag1_2024[df_small_lag1_2024.index.get_level_values('vat_num').isin(vat_nums)]

df_small_2024 = df_small[df_small.index.get_level_values('year') == 2024]
df_small_2024_right_nums = df_small_2024[df_small_2024.index.get_level_values('vat_num').isin(vat_nums)]


Длина датасета: 19979
Распределение таргета:
dflt_year
0    18668
1     1311
Name: count, dtype: int64
Длина датасета после добавления лаг фичей и отчистки строк с отсутствующими лаг-значениями: 5782
Распределение таргета:
target
0    5052
1     730
Name: count, dtype: int64
Длина датасета после добавления лаг фичей и отчистки строк с отсутствующими лаг-значениями: 12413
Распределение таргета:
target
0    11356
1     1057
Name: count, dtype: int64


In [42]:
df_big = prepare_dataset('../trading_df.csv',
                    exclude_cols=['cred_limit', 'fin_cond_index', 'tax_regime', 'reg_date', 'Unnamed: 0'], 
                    year_col='year',
                    id_col='vat_num',
                    dflt_col='dflt_year',
                    drop_fin_zeroes=True,
                    drop_ones_after_ones=True,
                    year_from=2022)

df_big_lag2 = make_lags_dataset(df_big, [1, 2])
df_big_lag2_right_nums = df_big_lag2[df_big_lag2.index.get_level_values('vat_num').isin(vat_nums)]

df_big_lag1 = make_lags_dataset(df_big, [1])
df_big_lag1_2024 = df_big_lag1[df_big_lag1.index.get_level_values('year') == 2024]
df_big_lag1_2024_right_nums = df_big_lag1_2024[df_big_lag1_2024.index.get_level_values('vat_num').isin(vat_nums)]

df_big_2024 = df_big[df_big.index.get_level_values('year') == 2024]
df_big_2024_right_nums = df_big_2024[df_big_2024.index.get_level_values('vat_num').isin(vat_nums)]

/home/jovyan/дефолт/inference/utils/common.py:42: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(df_path)


Длина датасета: 165303
Распределение таргета:
dflt_year
0    163992
1      1311
Name: count, dtype: int64
Длина датасета после добавления лаг фичей и отчистки строк с отсутствующими лаг-значениями: 51113
Распределение таргета:
target
0    50383
1      730
Name: count, dtype: int64
Длина датасета после добавления лаг фичей и отчистки строк с отсутствующими лаг-значениями: 106725
Распределение таргета:
target
0    105668
1      1057
Name: count, dtype: int64


Сравнение всех моделей на одном датасете (на одном множестве контрагентов, только по 2024г)

In [50]:
from utils.exp_handler import load_exp

loaded_big = load_exp('exp/big_df_no_lags')
loaded_big_lag1 = load_exp('exp/big_df_lag1')
loaded_big_lag2 = load_exp('exp/big_df_lag2')

loaded_small = load_exp('exp/small_df_no_lags')
loaded_small_lag1 = load_exp('exp/small_df_lag1')
loaded_small_lag2 = load_exp('exp/small_df_lag2')

In [61]:
print('Обученная на большом, lag=0')
calculate_metrics(loaded_big['calibrated_model'], df_big_2024_right_nums.drop(columns=['target']), df_big_2024_right_nums['target'])
print('Обученная на большом, lag=1')
calculate_metrics(loaded_big_lag1['calibrated_model'], df_big_lag1_2024_right_nums.drop(columns=['target']), df_big_lag1_2024_right_nums['target'])
print('Обученная на большом, lag=2')
calculate_metrics(loaded_big_lag2['calibrated_model'], df_big_lag2_right_nums.drop(columns=['target']), df_big_lag2_right_nums['target'])
pass

Обученная на большом, lag=0
AUC: 0.9237023991583422
Brier: 0.10657974968605025
LogLoss: 0.46481622803087036
Обученная на большом, lag=1
AUC: 0.93999040119741
Brier: 0.10180052953202415
LogLoss: 0.42687088897722886
Обученная на большом, lag=2
AUC: 0.9313095857872645
Brier: 0.10400808740392513
LogLoss: 0.4478956157084368


In [62]:
print('Обученная на меньшем, lag=0')
calculate_metrics(loaded_small['calibrated_model'], df_small_2024_right_nums.drop(columns=['target']), df_small_2024_right_nums['target'])
print('Обученная на меньшем, lag=1')
calculate_metrics(loaded_small_lag1['calibrated_model'], df_small_lag1_2024_right_nums.drop(columns=['target']), df_small_lag1_2024_right_nums['target'])
print('Обученная на меньшем, lag=2')
calculate_metrics(loaded_small_lag2['calibrated_model'], df_small_lag2.drop(columns=['target']), df_small_lag2['target'])
pass

Обученная на меньшем, lag=0
AUC: 0.9172591351316175
Brier: 0.08802727334764592
LogLoss: 0.2922757966772726
Обученная на меньшем, lag=1
AUC: 0.9373434093645268
Brier: 0.08267647666092855
LogLoss: 0.2800793333240446
Обученная на меньшем, lag=2
AUC: 0.9405493823143418
Brier: 0.08218195452457237
LogLoss: 0.28067610622239686


In [76]:
example = df_big_2024[df_big_2024['target'] == 1].iloc[[0]].reset_index()
example.to_excel('test_example.xlsx', index=False)